# TMW Hyperparameter Ablation Study — UCB Normal (PTB-DB)
This notebook performs an **ablation study** over three TMW hyperparameters:
- **`lambda_ts`**: Weight of the time-series loss in the generator loss
- **`mask_type`**: Mask computation strategy (1 = index-based, 2 = derivative-based)
- **`eps_threshold`**: Threshold for the binary mask matrix

Training uses a **two-stage schedule**:
1. Train a **shared base GAN** with `lambda_ts=0` for 50 epochs.
2. For each ablation combo, initialize from that base model and continue training to the target total epochs.

Unlike a full grid search, each hyperparameter is swept **one at a time** while the
remaining two are held at their **base (default) values**.

Results (checkpoints + metrics) are saved into `ablation_results/`.

In [1]:
import os
import sys
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import torch.autograd as autograd
from torch.utils.data import Dataset, DataLoader

from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce

# Add parent directory to path for imports
%cd ..
try:
    from losses import TimeSeriesLoss
    print("Successfully imported TimeSeriesLoss from losses.py")
except ImportError:
    print("Could not import losses.py. Make sure you are in the correct directory.")
%cd test_ucb_ablation_type_1
print(f"Current working directory: {os.getcwd()}")

n:\Fun\TMW
Successfully imported TimeSeriesLoss from losses.py
n:\Fun\TMW\test_ucb_ablation_type_1
Current working directory: n:\Fun\TMW\test_ucb_ablation_type_1


In [2]:
# ============================================================================
# ABLATION STUDY CONFIGURATION
# ============================================================================

# Base (default) values — held fixed when sweeping other hyperparameters
BASE = {
    "lambda_ts":     0.5,
    "mask_type":     1,
    "eps_threshold": 0.2,
}

# Stage-1 warm-start setup: train one shared base GAN without TS loss
BASE_PRETRAIN_EPOCHS = 50
BASE_PRETRAIN_CONFIG = {
    "lambda_ts": 0.0,
    "mask_type": BASE["mask_type"],
    "eps_threshold": BASE["eps_threshold"],
}

# Sweep ranges for each hyperparameter (tested one at a time)
# SWEEP = {
#     "lambda_ts":     [0, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10],
#     "mask_type":     [1],
#     "eps_threshold": [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 1],
# }

# Sweep ranges for each hyperparameter (tested one at a time)
SWEEP = {
    "lambda_ts":     [0.05, 0.1],
    "mask_type":     [1],
    "eps_threshold": [0.2],
}

# Build ablation combos: sweep one hyperparameter at a time
COMBOS = []  # list of (lambda_ts, mask_type, eps_threshold, swept_param_name)
seen = set()

for eps in SWEEP["eps_threshold"]:
    key = (BASE["lambda_ts"], BASE["mask_type"], eps)
    if key not in seen:
        seen.add(key)
        COMBOS.append((*key, "eps_threshold"))

for lts in SWEEP["lambda_ts"]:
    key = (lts, BASE["mask_type"], BASE["eps_threshold"])
    if key not in seen:
        seen.add(key)
        COMBOS.append((*key, "lambda_ts"))

for mt in SWEEP["mask_type"]:
    key = (BASE["lambda_ts"], mt, BASE["eps_threshold"])
    if key not in seen:
        seen.add(key)
        COMBOS.append((*key, "mask_type"))

print(f"Base configuration: {BASE}")
print(f"Base pretrain setup (shared init): {BASE_PRETRAIN_CONFIG}, epochs={BASE_PRETRAIN_EPOCHS}")
print(f"Total unique ablation runs: {len(COMBOS)}")
print()
for i, (lts, mt, eps, swept) in enumerate(COMBOS):
    marker = " <-- base" if (lts == BASE["lambda_ts"] and mt == BASE["mask_type"] and eps == BASE["eps_threshold"]) else ""
    print(f"  [{i+1:>3}] sweep={swept:<15s}  lambda_ts={lts}, mask_type={mt}, eps_threshold={eps}{marker}")

# Set to True to skip training and only evaluate existing checkpoints
EVAL_ONLY = False

# Base training configuration (shared across all combos)
TRAIN_CONFIG = {
    "epochs": 1000,
    "batch_size": 1024,
    "g_lr": 3e-4,
    "c_lr": 1e-3,
    "latent_dim": 100,
    "seq_len": 50,
    "channels": 1,
    "patch_size": 10,
    "n_critic": 3,
    "lambda_gp": 10,
    "lambda_sm": 1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_interval": 50,
    "num_eval_samples": 1000,
    "data_path": "../consolidated_datasets/pcb/ptbdb_normal.csv",
    "output_dir": "ablation_results",
}

# Fixed TMW hyperparameters (not swept)
TMW_FIXED_PARAMS = {
    "cost_function": "L2",
    "reg": 1,
    "max_iterations": 50,
    "thres": 1e-5,
    "masked": True,
    "rescale": True,
    "device": TRAIN_CONFIG["device"],
}

os.makedirs(TRAIN_CONFIG["output_dir"], exist_ok=True)
print(f"\nDevice: {TRAIN_CONFIG['device']}")
print(
    f"Training schedule: shared base pretrain for {BASE_PRETRAIN_EPOCHS} epochs, "
    f"then fine-tune each ablation run to total {TRAIN_CONFIG['epochs']} epochs."
)

Base configuration: {'lambda_ts': 0.5, 'mask_type': 1, 'eps_threshold': 0.2}
Base pretrain setup (shared init): {'lambda_ts': 0.0, 'mask_type': 1, 'eps_threshold': 0.2}, epochs=50
Total unique ablation runs: 3

  [  1] sweep=eps_threshold    lambda_ts=0.5, mask_type=1, eps_threshold=0.2 <-- base
  [  2] sweep=lambda_ts        lambda_ts=0.05, mask_type=1, eps_threshold=0.2
  [  3] sweep=lambda_ts        lambda_ts=0.1, mask_type=1, eps_threshold=0.2

Device: cuda
Training schedule: shared base pretrain for 50 epochs, then fine-tune each ablation run to total 1000 epochs.


In [3]:
# ============================================================================
# MODEL DEFINITIONS
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)
        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class Gen_TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=5, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )))


class Gen_TransformerEncoder(nn.Sequential):
    def __init__(self, depth=8, **kwargs):
        super().__init__(*[Gen_TransformerEncoderBlock(**kwargs) for _ in range(depth)])


class Generator(nn.Module):
    def __init__(self, seq_len=150, patch_size=15, channels=3, num_classes=9, latent_dim=100,
                 embed_dim=10, depth=3, num_heads=5, forward_drop_rate=0.5, attn_drop_rate=0.5):
        super(Generator, self).__init__()
        self.channels = channels
        self.latent_dim = latent_dim
        self.seq_len = seq_len
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.depth = depth
        self.attn_drop_rate = attn_drop_rate
        self.forward_drop_rate = forward_drop_rate

        self.l1 = nn.Linear(self.latent_dim, self.seq_len * self.embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.seq_len, self.embed_dim))
        self.blocks = Gen_TransformerEncoder(
            depth=self.depth,
            emb_size=self.embed_dim,
            drop_p=self.attn_drop_rate,
            forward_drop_p=self.forward_drop_rate
        )
        self.deconv = nn.Sequential(nn.Conv2d(self.embed_dim, self.channels, 1, 1, 0))

    def forward(self, z):
        x = self.l1(z).view(-1, self.seq_len, self.embed_dim)
        x = x + self.pos_embed
        H, W = 1, self.seq_len
        x = self.blocks(x)
        x = x.reshape(x.shape[0], 1, x.shape[1], x.shape[2])
        output = self.deconv(x.permute(0, 3, 1, 2))
        output = output.view(-1, self.channels, H, W)
        return output


class Dis_TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size=100, num_heads=5, drop_p=0., forward_expansion=4, forward_drop_p=0.):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )))


class Dis_TransformerEncoder(nn.Sequential):
    def __init__(self, depth=8, **kwargs):
        super().__init__(*[Dis_TransformerEncoderBlock(**kwargs) for _ in range(depth)])


class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size=100, n_classes=2):
        super().__init__()
        self.clshead = nn.Sequential(
            Reduce('b n e -> b e', reduction='mean'),
            nn.LayerNorm(emb_size),
            nn.Linear(emb_size, n_classes)
        )

    def forward(self, x):
        return self.clshead(x)


class PatchEmbedding_Linear(nn.Module):
    def __init__(self, in_channels=21, patch_size=16, emb_size=100, seq_length=1024):
        super().__init__()
        self.projection = nn.Sequential(
            Rearrange('b c h (w s2) -> b w (h s2 c)', s2=patch_size),
            nn.Linear(patch_size * in_channels, emb_size)
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.positions = nn.Parameter(torch.randn((seq_length // patch_size) + 1, emb_size))

    def forward(self, x: Tensor) -> Tensor:
        b, _, _, _ = x.shape
        x = self.projection(x)
        cls_tokens = repeat(self.cls_token, '() n e -> b n e', b=b)
        x = torch.cat([cls_tokens, x], dim=1)
        x += self.positions
        return x


class Discriminator(nn.Sequential):
    def __init__(self, in_channels=3, patch_size=15, emb_size=50, seq_length=150, depth=3, n_classes=1, **kwargs):
        super().__init__(
            PatchEmbedding_Linear(in_channels, patch_size, emb_size, seq_length),
            Dis_TransformerEncoder(depth, emb_size=emb_size, drop_p=0.5, forward_drop_p=0.5, **kwargs),
            ClassificationHead(emb_size, n_classes)
        )

In [4]:
# ============================================================================
# DATASET
# ============================================================================

class TimeSeriesDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def load_dataset(data_path, seq_len=50):
    """Load the PCB dataset."""
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Dataset not found at {data_path}")
    df = pd.read_csv(data_path, header=None)
    data = df.iloc[:, 5:5+seq_len].to_numpy()
    data = data[:, np.newaxis, np.newaxis, :]
    data = torch.tensor(data).float()
    return TimeSeriesDataset(data)

# Load Data
try:
    dataset = load_dataset(TRAIN_CONFIG['data_path'], TRAIN_CONFIG['seq_len'])
    dataloader = DataLoader(dataset, batch_size=TRAIN_CONFIG['batch_size'], shuffle=True)
    print(f"Dataset loaded. Size: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")

Dataset loaded. Size: 4046


In [5]:
# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    """Calculates the gradient penalty loss for WGAN GP."""
    alpha = torch.rand(real_samples.size(0), 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    critic_interpolates = critic(interpolates)
    gradients = autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(critic_interpolates),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradients = gradients.reshape(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


def save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, epoch):
    """Save checkpoint, removing old ones to save space."""
    for f in os.listdir(ckpt_dir):
        if f.startswith("checkpoint_") and f.endswith(".pth"):
            os.remove(os.path.join(ckpt_dir, f))
    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'critic_state_dict': critic.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_C_state_dict': optimizer_C.state_dict(),
    }
    torch.save(checkpoint, os.path.join(ckpt_dir, f"checkpoint_latest.pth"))
    print(f"  [Checkpoint] Saved at epoch {epoch}")


def load_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, device):
    """Load checkpoint if exists."""
    ckpt_path = os.path.join(ckpt_dir, "checkpoint_latest.pth")
    if os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        generator.load_state_dict(checkpoint['generator_state_dict'])
        critic.load_state_dict(checkpoint['critic_state_dict'])
        optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
        optimizer_C.load_state_dict(checkpoint['optimizer_C_state_dict'])
        start_epoch = checkpoint['epoch']
        print(f"  [Checkpoint] Resumed from epoch {start_epoch}")
        return start_epoch
    return 0

In [6]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def combo_tag(lambda_ts, mask_type, eps_threshold):
    """Create a short, filesystem-safe tag for a hyperparameter combination."""
    return f"lts{lambda_ts}_mt{mask_type}_eps{eps_threshold}"


def load_model_weights_only(ckpt_dir, generator, critic, device):
    """Load only model weights from a checkpoint (no optimizer state)."""
    ckpt_path = os.path.join(ckpt_dir, "checkpoint_latest.pth")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    critic.load_state_dict(checkpoint['critic_state_dict'])
    return checkpoint.get('epoch', 0)


def train_single_combo(
    lambda_ts,
    mask_type,
    eps_threshold,
    config,
    dataloader,
    output_dir,
    init_ckpt_dir=None,
    target_epochs=None,
    run_tag=None,
    ckpt_dir=None,
):
    """Train GAN for one config; optionally initialize from a shared base checkpoint."""
    tag = run_tag if run_tag is not None else combo_tag(lambda_ts, mask_type, eps_threshold)
    final_epochs = config['epochs'] if target_epochs is None else target_epochs

    print(f"\n{'='*60}")
    print(f"Training TMW  |  lambda_ts={lambda_ts}  mask_type={mask_type}  eps_threshold={eps_threshold}")
    print(f"Tag: {tag}")
    print(f"Target epochs: {final_epochs}")
    print(f"{'='*60}")

    device = config['device']
    ckpt_dir = ckpt_dir if ckpt_dir is not None else os.path.join(output_dir, f"ckpt_{tag}")
    os.makedirs(ckpt_dir, exist_ok=True)

    # Initialize models
    generator = Generator(
        seq_len=config['seq_len'],
        patch_size=config['patch_size'],
        channels=config['channels'],
        latent_dim=config['latent_dim'],
        depth=3
    ).to(device)

    critic = Discriminator(
        in_channels=config['channels'],
        patch_size=config['patch_size'],
        seq_length=config['seq_len'],
        n_classes=1
    ).to(device)

    # Build TMW loss with the swept hyperparameters
    loss_params = TMW_FIXED_PARAMS.copy()
    loss_params["mask_type"] = mask_type
    loss_params["eps_threshold"] = eps_threshold
    ts_loss = TimeSeriesLoss("tmw", loss_params)
    smooth_loss = nn.L1Loss()

    # Optimizers
    optimizer_G = torch.optim.Adam(generator.parameters(), lr=config['g_lr'], betas=(0.0, 0.9))
    optimizer_C = torch.optim.Adam(critic.parameters(), lr=config['c_lr'], betas=(0.0, 0.9))

    # Resume this run if its own checkpoint exists.
    # Otherwise, warm-start from a shared base checkpoint if provided.
    start_epoch = 0
    run_ckpt_path = os.path.join(ckpt_dir, "checkpoint_latest.pth")
    if os.path.exists(run_ckpt_path):
        start_epoch = load_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, device)
    elif init_ckpt_dir is not None:
        init_epoch = load_model_weights_only(init_ckpt_dir, generator, critic, device)
        start_epoch = init_epoch
        print(f"  [Init] Loaded model weights from base checkpoint at epoch {init_epoch}")

    # Training loop
    critic_losses = []
    generator_losses = []
    critic_loss = torch.tensor(0.0, device=device)
    generator_loss = torch.tensor(0.0, device=device)

    if start_epoch >= final_epochs:
        print(f"  [Skip] Existing checkpoint already reached epoch {start_epoch} >= target {final_epochs}")

    for epoch in range(start_epoch, final_epochs):
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{final_epochs}", leave=False)

        for real_series in progress_bar:
            real_series = real_series.to(device)

            # Train Critic
            for _ in range(config['n_critic']):
                optimizer_C.zero_grad()
                noise = torch.randn(real_series.size(0), config['latent_dim'], device=device)
                fake_series = generator(noise).detach()

                real_scores = critic(real_series)
                fake_scores = critic(fake_series)

                loss_w = torch.mean(fake_scores) - torch.mean(real_scores)
                gradient_penalty = compute_gradient_penalty(critic, real_series, fake_series, device)
                critic_loss = loss_w + config['lambda_gp'] * gradient_penalty
                critic_loss.backward()
                optimizer_C.step()

            # Train Generator
            optimizer_G.zero_grad()
            noise = torch.randn(real_series.size(0), config['latent_dim'], device=device)
            fake_series_for_g = generator(noise)

            adversarial_loss = -torch.mean(critic(fake_series_for_g))

            # Time-Series Loss (reshape to batch, seq_len, dim)
            fake_reshaped = fake_series_for_g.squeeze(2).permute(0, 2, 1)
            real_reshaped = real_series.squeeze(2).permute(0, 2, 1)

            if lambda_ts > 0:
                ts_loss_val = ts_loss(fake_reshaped, real_reshaped)
                if isinstance(ts_loss_val, tuple):
                    ts_loss_val = ts_loss_val[0]
                ts_loss_mean = ts_loss_val.mean() if ts_loss_val.dim() > 0 else ts_loss_val
            else:
                ts_loss_mean = torch.tensor(0.0, device=device)

            l2_smooth_loss = smooth_loss(fake_series_for_g[:, :, :, 1:], fake_series_for_g[:, :, :, :-1])

            generator_loss = adversarial_loss + lambda_ts * ts_loss_mean + config['lambda_sm'] * l2_smooth_loss
            generator_loss.backward()
            optimizer_G.step()

            progress_bar.set_postfix({
                "C_Loss": f"{critic_loss.item():.4f}",
                "G_Loss": f"{generator_loss.item():.4f}"
            })

        critic_losses.append(float(critic_loss.item()))
        generator_losses.append(float(generator_loss.item()))

        # Save checkpoint
        if (epoch + 1) % config['save_interval'] == 0:
            save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, epoch + 1)

    # Save final model
    save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, final_epochs)
    torch.save(generator.state_dict(), os.path.join(ckpt_dir, "generator_final.pth"))

    # Save loss plot only if this run performed at least one epoch in this call
    if generator_losses and critic_losses:
        plt.figure(figsize=(10, 5))
        plt.title(f"Training Losses - TMW [{tag}]")
        plt.plot(generator_losses, label="Generator")
        plt.plot(critic_losses, label="Critic")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(output_dir, f"{tag}_training_loss.png"), dpi=150)
        plt.close()

    print(f"Training completed for TMW [{tag}]")
    return generator

In [7]:
# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

def generate_samples(generator, num_samples, latent_dim, device, seq_len, channels):
    """Generate fake samples from the generator."""
    generator.eval()
    fake_samples = []
    with torch.no_grad():
        for _ in range(num_samples):
            noise = torch.randn(1, latent_dim, device=device)
            fake_sample = generator(noise).cpu()
            fake_samples.append(fake_sample)
    fake_samples = torch.cat(fake_samples, dim=0)
    return fake_samples.squeeze().reshape(num_samples, seq_len, channels).numpy()


def feature_extract(dist_fake, dist_real):
    """Extract features using MiniRocket."""
    from sktime.transformations.panel.rocket import MiniRocketMultivariate
    dist = np.concatenate((dist_fake, dist_real), axis=0)
    dist = dist.transpose(0, 2, 1)
    rocket = MiniRocketMultivariate()
    features = rocket.fit_transform(dist)
    features_fake = features[:len(dist_fake)]
    features_real = features[len(dist_fake):]
    return features_fake, features_real


def compute_mmd(features_fake, features_real, device='cpu'):
    """Compute Maximum Mean Discrepancy."""
    dist_fake = torch.tensor(features_fake.to_numpy() if hasattr(features_fake, 'to_numpy') else features_fake,
                             dtype=torch.float32).to(device)
    dist_real = torch.tensor(features_real.to_numpy() if hasattr(features_real, 'to_numpy') else features_real,
                             dtype=torch.float32).to(device)

    xx = torch.matmul(dist_fake, dist_fake.t())
    yy = torch.matmul(dist_real, dist_real.t())
    zz = torch.matmul(dist_fake, dist_real.t())

    rx = xx.diag().unsqueeze(0).expand_as(xx)
    ry = yy.diag().unsqueeze(0).expand_as(yy)

    dxx = rx.t() + rx - 2. * xx
    dyy = ry.t() + ry - 2. * yy
    dxy = rx.t() + ry - 2. * zz

    all_distances = torch.cat([dxx.flatten(), dyy.flatten(), dxy.flatten()])
    median_sq_dist = torch.median(all_distances).clamp(min=1e-6)
    sigma = torch.sqrt(median_sq_dist / 2.0)

    XX = torch.exp(-dxx / (2.0 * sigma**2))
    YY = torch.exp(-dyy / (2.0 * sigma**2))
    XY = torch.exp(-dxy / (2.0 * sigma**2))

    return (torch.mean(XX) + torch.mean(YY) - 2.0 * torch.mean(XY)).item()


def matrix_sqrt(matrix):
    """Compute the matrix square root using eigendecomposition."""
    matrix = (matrix + matrix.T) / 2
    eigenvalues, eigenvectors = torch.linalg.eigh(matrix)
    eigenvalues = torch.clamp(eigenvalues, min=0)
    sqrt_eigenvalues = torch.sqrt(eigenvalues)
    sqrt_matrix = eigenvectors @ torch.diag(sqrt_eigenvalues) @ eigenvectors.T
    return sqrt_matrix


def compute_frechet_distance(features_fake, features_real, device='cpu'):
    """Compute Frechet Distance (FID-like) using ROCKET features."""
    if hasattr(features_fake, 'to_numpy'):
        feat_fake = torch.tensor(features_fake.to_numpy(), dtype=torch.float64, device=device)
    else:
        feat_fake = torch.tensor(features_fake, dtype=torch.float64, device=device)
    if hasattr(features_real, 'to_numpy'):
        feat_real = torch.tensor(features_real.to_numpy(), dtype=torch.float64, device=device)
    else:
        feat_real = torch.tensor(features_real, dtype=torch.float64, device=device)

    mu_real = torch.mean(feat_real, dim=0)
    mu_fake = torch.mean(feat_fake, dim=0)

    feat_real_centered = feat_real - mu_real
    feat_fake_centered = feat_fake - mu_fake

    n_real = feat_real.shape[0]
    n_fake = feat_fake.shape[0]

    sigma_real = (feat_real_centered.T @ feat_real_centered) / (n_real - 1)
    sigma_fake = (feat_fake_centered.T @ feat_fake_centered) / (n_fake - 1)

    diff = mu_real - mu_fake

    try:
        product = sigma_real @ sigma_fake
        covmean = matrix_sqrt(product)
    except Exception:
        offset = torch.eye(sigma_real.shape[0], dtype=torch.float64, device=device) * 1e-6
        product = (sigma_real + offset) @ (sigma_fake + offset)
        covmean = matrix_sqrt(product)

    fd = torch.dot(diff, diff) + torch.trace(sigma_real) + torch.trace(sigma_fake) - 2 * torch.trace(covmean)
    return float(fd.item())


# ============================================================================
# ACS and AJSD Metrics (TTS-GAN protocol)
# ============================================================================

def extract_statistical_features(data):
    """Extract 7 statistical features per channel."""
    from scipy.stats import skew, kurtosis
    N, T, C = data.shape
    features = np.zeros((N, 7 * C))
    for i in range(N):
        for c in range(C):
            seq = data[i, :, c]
            features[i, c * 7 + 0] = np.mean(seq)
            features[i, c * 7 + 1] = np.var(seq)
            features[i, c * 7 + 2] = skew(seq)
            features[i, c * 7 + 3] = kurtosis(seq)
            features[i, c * 7 + 4] = np.min(seq)
            features[i, c * 7 + 5] = np.max(seq)
            features[i, c * 7 + 6] = np.median(seq)
    return features


def compute_acs(real_data, fake_data):
    """Compute Average Cosine Similarity (ACS). Closer to 1 is better."""
    feat_real = extract_statistical_features(real_data)
    feat_fake = extract_statistical_features(fake_data)
    N = min(len(feat_real), len(feat_fake))
    similarities = []
    for i in range(N):
        norm_real = np.linalg.norm(feat_real[i])
        norm_fake = np.linalg.norm(feat_fake[i])
        if norm_real > 1e-8 and norm_fake > 1e-8:
            sim = np.dot(feat_real[i], feat_fake[i]) / (norm_real * norm_fake)
            similarities.append(sim)
    return float(np.mean(similarities)) if similarities else 0.0


def compute_ajsd(real_data, fake_data, n_bins=50):
    """Compute Average Jensen-Shannon Distance (AJSD). Closer to 0 is better."""
    from scipy.spatial.distance import jensenshannon
    feat_real = extract_statistical_features(real_data)
    feat_fake = extract_statistical_features(fake_data)
    D = feat_real.shape[1]
    jsd_values = []
    for d in range(D):
        all_vals = np.concatenate([feat_real[:, d], feat_fake[:, d]])
        min_val, max_val = np.min(all_vals), np.max(all_vals)
        if max_val - min_val < 1e-8:
            continue
        bins = np.linspace(min_val, max_val, n_bins + 1)
        hist_real, _ = np.histogram(feat_real[:, d], bins=bins, density=True)
        hist_fake, _ = np.histogram(feat_fake[:, d], bins=bins, density=True)
        hist_real = hist_real + 1e-10
        hist_fake = hist_fake + 1e-10
        hist_real = hist_real / hist_real.sum()
        hist_fake = hist_fake / hist_fake.sum()
        jsd = jensenshannon(hist_real, hist_fake)
        jsd_values.append(jsd)
    return float(np.mean(jsd_values)) if jsd_values else 0.0


# ============================================================================
# Discriminative and Predictive Scores (TimeGAN protocol)
# ============================================================================

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return torch.sigmoid(out)


class LSTMPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)


def compute_discriminative_score(real_data, fake_data, device='cpu', epochs=500, batch_size=256):
    """Discriminative Score = |accuracy - 0.5|, closer to 0 is better."""
    from torch.utils.data import TensorDataset, DataLoader
    real_tensor = torch.tensor(real_data, dtype=torch.float32)
    fake_tensor = torch.tensor(fake_data, dtype=torch.float32)
    real_labels = torch.ones(len(real_data), 1)
    fake_labels = torch.zeros(len(fake_data), 1)
    all_data = torch.cat([real_tensor, fake_tensor], dim=0)
    all_labels = torch.cat([real_labels, fake_labels], dim=0)
    n_samples = len(all_data)
    indices = torch.randperm(n_samples)
    train_size = int(0.8 * n_samples)
    train_data = all_data[indices[:train_size]]
    train_labels = all_labels[indices[:train_size]]
    test_data = all_data[indices[train_size:]]
    test_labels = all_labels[indices[train_size:]]
    train_loader = DataLoader(TensorDataset(train_data, train_labels), batch_size=batch_size, shuffle=True)
    input_dim = real_data.shape[2]
    model = LSTMClassifier(input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        test_data = test_data.to(device)
        test_labels = test_labels.to(device)
        pred = model(test_data)
        pred_labels = (pred > 0.5).float()
        accuracy = (pred_labels == test_labels).float().mean().item()
    return abs(accuracy - 0.5)


def compute_predictive_score(real_data, fake_data, device='cpu', epochs=500, batch_size=256):
    """Predictive Score (TSTR MAE), lower is better."""
    from torch.utils.data import TensorDataset, DataLoader
    fake_tensor = torch.tensor(fake_data, dtype=torch.float32)
    fake_input = fake_tensor[:, :-1, :]
    fake_target = fake_tensor[:, 1:, :]
    train_loader = DataLoader(TensorDataset(fake_input, fake_target), batch_size=batch_size, shuffle=True)
    input_dim = real_data.shape[2]
    model = LSTMPredictor(input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
    real_tensor = torch.tensor(real_data, dtype=torch.float32)
    real_input = real_tensor[:, :-1, :]
    real_target = real_tensor[:, 1:, :]
    model.eval()
    with torch.no_grad():
        real_input = real_input.to(device)
        real_target = real_target.to(device)
        pred = model(real_input)
        mae = torch.mean(torch.abs(pred - real_target)).item()
    return mae

    
def evaluate_model(tag, generator, real_dataset, config, output_dir, num_runs=10):
    """Evaluate a trained model and return metrics dict with mean and std."""
    print(f"\n  Evaluating TMW [{tag}] over {num_runs} runs...")
    device = config['device']
    num_samples = config['num_eval_samples']
    seq_len = config['seq_len']
    channels = config['channels']
    real_samples = real_dataset[:num_samples].squeeze().reshape(num_samples, seq_len, channels).numpy()

    all_mmd, all_fd, all_acs, all_ajsd, all_disc, all_pred = [], [], [], [], [], []

    for run_idx in range(num_runs):
        print(f"    Run {run_idx + 1}/{num_runs}...")
        fake_samples = generate_samples(generator, num_samples, config['latent_dim'], device, seq_len, channels)
        features_fake, features_real = feature_extract(fake_samples, real_samples)

        mmd_score = compute_mmd(features_fake, features_real, device)
        fd_score = compute_frechet_distance(features_fake, features_real, device)
        acs_score = compute_acs(real_samples, fake_samples)
        ajsd_score = compute_ajsd(real_samples, fake_samples)
        # disc_score = compute_discriminative_score(real_samples, fake_samples, device)
        # pred_score = compute_predictive_score(real_samples, fake_samples, device)
        disc_score = 0.0  # Placeholder since these are expensive to compute
        pred_score = 0.0  # Placeholder since these are expensive to compute

        all_mmd.append(mmd_score)
        all_fd.append(fd_score)
        all_acs.append(acs_score)
        all_ajsd.append(ajsd_score)
        all_disc.append(disc_score)
        all_pred.append(pred_score)

        print(f"      MMD={mmd_score:.6f}  FD={fd_score:.2f}  ACS={acs_score:.4f}  AJSD={ajsd_score:.4f}  Disc={disc_score:.4f}  Pred={pred_score:.4f}")

    return {
        "tag": tag,
        "MMD":  (np.mean(all_mmd),  np.std(all_mmd)),
        "FD":   (np.mean(all_fd),   np.std(all_fd)),
        "ACS":  (np.mean(all_acs),  np.std(all_acs)),
        "AJSD": (np.mean(all_ajsd), np.std(all_ajsd)),
        "Disc": (np.mean(all_disc), np.std(all_disc)),
        "Pred": (np.mean(all_pred), np.std(all_pred)),
        "num_runs": num_runs,
    }

In [ ]:
# ============================================================================
# MAIN EXECUTION — ablation: shared base pretrain + sweep one hyperparameter
# ============================================================================

all_metrics = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = TRAIN_CONFIG["output_dir"]
os.makedirs(output_dir, exist_ok=True)

# Shared stage-1 base pretraining (without time-series loss)
base_ckpt_dir = os.path.join(output_dir, "ckpt_base_pretrain")

if not EVAL_ONLY:
    print("\n>>> Stage 1/2: Training shared base GAN (no TS loss)")
    _ = train_single_combo(
        lambda_ts=BASE_PRETRAIN_CONFIG["lambda_ts"],
        mask_type=BASE_PRETRAIN_CONFIG["mask_type"],
        eps_threshold=BASE_PRETRAIN_CONFIG["eps_threshold"],
        config=TRAIN_CONFIG,
        dataloader=dataloader,
        output_dir=output_dir,
        init_ckpt_dir=None,
        target_epochs=BASE_PRETRAIN_EPOCHS,
        run_tag="base_pretrain",
        ckpt_dir=base_ckpt_dir,
    )
else:
    if not os.path.exists(os.path.join(base_ckpt_dir, "checkpoint_latest.pth")):
        print("Warning: EVAL_ONLY=True but base pretrain checkpoint not found.")
        print("Ablation models can still be evaluated if their individual checkpoints exist.")

print("\n>>> Stage 2/2: Ablation fine-tuning from the shared base model")

for idx, (lambda_ts, mask_type, eps_threshold, swept_param) in enumerate(COMBOS):
    tag = combo_tag(lambda_ts, mask_type, eps_threshold)
    print(f"\n>>> Ablation run {idx+1}/{len(COMBOS)}: {tag}  (sweeping {swept_param})")

    current_config = TRAIN_CONFIG.copy()

    if not EVAL_ONLY:
        generator = train_single_combo(
            lambda_ts=lambda_ts,
            mask_type=mask_type,
            eps_threshold=eps_threshold,
            config=current_config,
            dataloader=dataloader,
            output_dir=output_dir,
            init_ckpt_dir=base_ckpt_dir,
            target_epochs=current_config['epochs'],
        )
    else:
        ckpt_path = os.path.join(output_dir, f"ckpt_{tag}", "generator_final.pth")
        if not os.path.exists(ckpt_path):
            print(f"  Warning: No trained model found for {tag}, skipping...")
            continue
        generator = Generator(
            seq_len=current_config['seq_len'],
            patch_size=current_config['patch_size'],
            channels=current_config['channels'],
            latent_dim=current_config['latent_dim'],
            depth=3
        ).to(current_config['device'])
        generator.load_state_dict(torch.load(ckpt_path, map_location=current_config['device']))

    metrics = evaluate_model(tag, generator, dataset, current_config, output_dir, num_runs=10)
    metrics["lambda_ts"] = lambda_ts
    metrics["mask_type"] = mask_type
    metrics["eps_threshold"] = eps_threshold
    metrics["swept_param"] = swept_param
    all_metrics.append(metrics)

    # --- live-save after each combo so nothing is lost on crash ---
    _rows = []
    for m in all_metrics:
        _rows.append({
            "swept_param": m["swept_param"],
            "lambda_ts": m["lambda_ts"],
            "mask_type": m["mask_type"],
            "eps_threshold": m["eps_threshold"],
            "MMD_mean": m["MMD"][0], "MMD_std": m["MMD"][1],
            "FD_mean": m["FD"][0], "FD_std": m["FD"][1],
            "ACS_mean": m["ACS"][0], "ACS_std": m["ACS"][1],
            "AJSD_mean": m["AJSD"][0], "AJSD_std": m["AJSD"][1],
            "Disc_mean": m["Disc"][0], "Disc_std": m["Disc"][1],
            "Pred_mean": m["Pred"][0], "Pred_std": m["Pred"][1],
        })
    pd.DataFrame(_rows).to_csv(os.path.join(output_dir, f"ablation_results_{timestamp}.csv"), index=False)

print("\n\nAll ablation runs finished!")


>>> Stage 1/2: Training shared base GAN (no TS loss)

Training TMW  |  lambda_ts=0.0  mask_type=1  eps_threshold=0.2
Tag: base_pretrain
Target epochs: 50
  [Checkpoint] Resumed from epoch 50
  [Skip] Existing checkpoint already reached epoch 50 >= target 50
  [Checkpoint] Saved at epoch 50
Training completed for TMW [base_pretrain]

>>> Stage 2/2: Ablation fine-tuning from the shared base model

>>> Ablation run 1/3: lts0.5_mt1_eps0.2  (sweeping eps_threshold)

Training TMW  |  lambda_ts=0.5  mask_type=1  eps_threshold=0.2
Tag: lts0.5_mt1_eps0.2
Target epochs: 1000
  [Checkpoint] Resumed from epoch 1000
  [Skip] Existing checkpoint already reached epoch 1000 >= target 1000
  [Checkpoint] Saved at epoch 1000
Training completed for TMW [lts0.5_mt1_eps0.2]

  Evaluating TMW [lts0.5_mt1_eps0.2] over 10 runs...
    Run 1/10...


In [ ]:
# ============================================================================
# RESULTS SUMMARY
# ============================================================================

results_df = pd.read_csv(os.path.join(output_dir, f"ablation_results_{timestamp}.csv"))
print(f"Total results: {len(results_df)} ablation runs\n")

# ---- Pretty-print results grouped by swept parameter ----
for swept_name in ["lambda_ts", "mask_type", "eps_threshold"]:
    sub = results_df[results_df['swept_param'] == swept_name]
    if len(sub) == 0:
        continue
    print("=" * 160)
    print(f"ABLATION: Sweeping  {swept_name}   (other params fixed at base: {BASE})")
    print("=" * 160)
    header = f"{'lambda_ts':>10} {'mask_type':>10} {'eps_threshold':>14} | {'MMD':>20} | {'FD':>20} | {'ACS':>16} | {'AJSD':>16} | {'Disc':>16} | {'Pred':>16}"
    print(header)
    print("-" * 160)
    for _, row in sub.iterrows():
        print(
            f"{row['lambda_ts']:>10} {int(row['mask_type']):>10} {row['eps_threshold']:>14} | "
            f"{row['MMD_mean']:>9.6f}\u00b1{row['MMD_std']:<9.6f} | "
            f"{row['FD_mean']:>9.2f}\u00b1{row['FD_std']:<9.2f} | "
            f"{row['ACS_mean']:>7.4f}\u00b1{row['ACS_std']:<7.4f} | "
            f"{row['AJSD_mean']:>7.4f}\u00b1{row['AJSD_std']:<7.4f} | "
            f"{row['Disc_mean']:>7.4f}\u00b1{row['Disc_std']:<7.4f} | "
            f"{row['Pred_mean']:>7.4f}\u00b1{row['Pred_std']:<7.4f}"
        )
    print()

# ---- Best combo per metric ----
print("\n--- Best Configuration per Metric (across all ablation runs) ---")
best_metrics = {
    "MMD  (\u2193)": results_df.loc[results_df['MMD_mean'].idxmin()],
    "FD   (\u2193)": results_df.loc[results_df['FD_mean'].idxmin()],
    "ACS  (\u2191)": results_df.loc[results_df['ACS_mean'].idxmax()],
    "AJSD (\u2193)": results_df.loc[results_df['AJSD_mean'].idxmin()],
    "Disc (\u2193)": results_df.loc[results_df['Disc_mean'].idxmin()],
    "Pred (\u2193)": results_df.loc[results_df['Pred_mean'].idxmin()],
}
for metric_label, best_row in best_metrics.items():
    print(f"  {metric_label}: lambda_ts={best_row['lambda_ts']}, mask_type={int(best_row['mask_type'])}, eps_threshold={best_row['eps_threshold']}")

In [ ]:
# ============================================================================
# LINE-PLOT VISUALIZATIONS — one subplot per swept param, one line per metric
# ============================================================================

metric_cols = ["MMD_mean", "FD_mean", "ACS_mean", "AJSD_mean", "Disc_mean", "Pred_mean"]
metric_labels = ["MMD (\u2193)", "FD (\u2193)", "ACS (\u2191)", "AJSD (\u2193)", "Disc (\u2193)", "Pred (\u2193)"]

swept_params = ["lambda_ts", "mask_type", "eps_threshold"]

for metric_col, metric_label in zip(metric_cols, metric_labels):
    fig, axes = plt.subplots(1, len(swept_params), figsize=(6 * len(swept_params), 4))
    fig.suptitle(f"Ablation Study — {metric_label}", fontsize=14)

    for ax, sp in zip(axes, swept_params):
        sub = results_df[results_df['swept_param'] == sp].sort_values(sp)
        if len(sub) == 0:
            ax.set_title(f"Sweep: {sp} (no data)")
            continue

        std_col = metric_col.replace('_mean', '_std')
        x_vals = sub[sp].values
        y_vals = sub[metric_col].values
        y_std = sub[std_col].values

        ax.errorbar(range(len(x_vals)), y_vals, yerr=y_std, marker='o', capsize=4)
        ax.set_xticks(range(len(x_vals)))
        ax.set_xticklabels([str(v) for v in x_vals], rotation=45)
        ax.set_xlabel(sp)
        ax.set_ylabel(metric_label)
        ax.set_title(f"Sweep: {sp}")
        ax.grid(True, alpha=0.3)

        # Highlight base value
        base_val = BASE[sp]
        if base_val in x_vals:
            base_idx = list(x_vals).index(base_val)
            ax.axvline(x=base_idx, color='red', linestyle='--', alpha=0.5, label=f'base={base_val}')
            ax.legend(fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(os.path.join(output_dir, f"ablation_{metric_col}.png"), dpi=150)
    plt.show()

print("\nAblation plots saved to", output_dir)

In [ ]:
# ============================================================================
# SAVE FINAL SUMMARY TEXT FILE
# ============================================================================

summary_path = os.path.join(output_dir, f"ablation_summary_{timestamp}.txt")
with open(summary_path, "w") as f:
    f.write(f"TMW Hyperparameter Ablation Study Results — {timestamp}\n")
    f.write(f"Dataset: PTB-DB Normal (UCB)\n")
    f.write(f"Base configuration: {BASE}\n")
    f.write(f"Shared base pretrain setup: {BASE_PRETRAIN_CONFIG}\n")
    f.write(f"Base pretrain epochs: {BASE_PRETRAIN_EPOCHS}\n")
    f.write(f"Swept hyperparameters (one at a time):\n")
    for k, v in SWEEP.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"Total unique ablation runs: {len(COMBOS)}\n")
    f.write(f"Evaluation runs per combo: 10\n")
    f.write(f"Fine-tune total epochs per ablation run: {TRAIN_CONFIG['epochs']}\n\n")

    f.write("Metric Descriptions:\n")
    f.write("  MMD:  Maximum Mean Discrepancy (lower is better)\n")
    f.write("  FD:   Frechet Distance using ROCKET features (lower is better)\n")
    f.write("  ACS:  Average Cosine Similarity (higher is better, max 1.0)\n")
    f.write("  AJSD: Average Jensen-Shannon Distance (lower is better, min 0.0)\n")
    f.write("  Disc: Discriminative Score (lower is better, min 0.0)\n")
    f.write("  Pred: Predictive Score/MAE (lower is better)\n\n")

    for swept_name in ["lambda_ts", "mask_type", "eps_threshold"]:
        sub = results_df[results_df['swept_param'] == swept_name]
        if len(sub) == 0:
            continue
        f.write("=" * 160 + "\n")
        f.write(f"ABLATION: Sweeping  {swept_name}   (other params fixed at base: {BASE})\n")
        f.write("=" * 160 + "\n")
        f.write(f"{'lambda_ts':>10} {'mask_type':>10} {'eps_threshold':>14} | {'MMD':>20} | {'FD':>20} | {'ACS':>16} | {'AJSD':>16} | {'Disc':>16} | {'Pred':>16}\n")
        f.write("-" * 160 + "\n")
        for _, row in sub.iterrows():
            f.write(
                f"{row['lambda_ts']:>10} {int(row['mask_type']):>10} {row['eps_threshold']:>14} | "
                f"{row['MMD_mean']:>9.6f}±{row['MMD_std']:<9.6f} | "
                f"{row['FD_mean']:>9.2f}±{row['FD_std']:<9.2f} | "
                f"{row['ACS_mean']:>7.4f}±{row['ACS_std']:<7.4f} | "
                f"{row['AJSD_mean']:>7.4f}±{row['AJSD_std']:<7.4f} | "
                f"{row['Disc_mean']:>7.4f}±{row['Disc_std']:<7.4f} | "
                f"{row['Pred_mean']:>7.4f}±{row['Pred_std']:<7.4f}\n"
            )
        f.write("\n")

    f.write("\n--- Best Configuration per Metric (across all ablation runs) ---\n")
    for metric_label, best_row in best_metrics.items():
        f.write(f"  {metric_label}: lambda_ts={best_row['lambda_ts']}, mask_type={int(best_row['mask_type'])}, eps_threshold={best_row['eps_threshold']}\n")

print(f"Summary saved to: {summary_path}")